# 30 - Analyze source delayed refinement

This is the causal evaluation of **five unrefined prediction chunks, then refinement always on**.
It exact-matches every completed delayed-refinement rollout against the original K=5, Euler-steps
`(3,4)` uncertainty-only source rollout for the same suite, task, episode initialization, and
state hash. Every reported SR and SR delta uses the entire matched set—not only episodes that
survived until the sixth prediction chunk.

`REQUIRE_FULL_COHORT=False` permits an interim look while workers are running. Set it to `True`
for the final 1,300-episode result.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Fetch and exact-match delayed refinement with the source baseline

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from analysis.statistics import paired_bootstrap_ci, discordant_test
from pnp.config import Method
from pnp.diversity import (DIVERSITY_PAIR_KEYS,
    SOURCE_DELAYED_REFINEMENT_EXPERIMENT)
from pnp.experiments import PRO_EXPANDED_EXPERIMENT
from pnp.store import SupabaseStore

REFINE_START_CHUNK = 5
EXPECTED_EPISODES = 1300
REQUIRE_FULL_COHORT = False  # set True after workers 0,1,2,3 are complete
OUTPUT = Path("source_delayed_refinement_outputs")
OUTPUT.mkdir(exist_ok=True)
store = SupabaseStore()

delayed = pd.DataFrame(store.fetch_all(
    "rollouts", "*", configure=lambda query: query.eq(
        "experiment", SOURCE_DELAYED_REFINEMENT_EXPERIMENT).eq(
            "method", Method.DELAYED_REFINEMENT), order_by=("rollout_id",)))
source = pd.DataFrame(store.fetch_all(
    "rollouts", "*", configure=lambda query: query.eq(
        "experiment", PRO_EXPANDED_EXPERIMENT).eq(
            "method", Method.UNCERTAINTY), order_by=("rollout_id",)))
assert len(delayed), "No delayed-refinement rollouts found"

def start_chunk(value):
    if isinstance(value, str):
        value = json.loads(value)
    return (value or {}).get("refine_start_chunk")

delayed = delayed[delayed.status.eq("completed")].copy()
delayed["refine_start_chunk"] = delayed.config_json.apply(start_chunk)
delayed = delayed[
    delayed.refine_start_chunk.eq(REFINE_START_CHUNK)
    & delayed.pnp_k.eq(5)
    & delayed.pnp_step_indices.apply(lambda value: tuple(value or []) == (3, 4))].copy()
baseline = source[
    source.status.eq("completed") & source.pnp_k.eq(5)
    & source.pnp_step_indices.apply(lambda value: tuple(value or []) == (3, 4))].copy()
for name, frame in (("delayed", delayed), ("baseline", baseline)):
    assert not frame.duplicated(DIVERSITY_PAIR_KEYS).any(), f"duplicate {name} identities"

paired = (delayed[DIVERSITY_PAIR_KEYS + ["rollout_id", "success", "n_steps", "n_chunks"]]
          .rename(columns={"rollout_id": "delayed_rollout_id",
                           "success": "delayed_success",
                           "n_steps": "delayed_n_steps",
                           "n_chunks": "delayed_n_chunks"})
          .merge(baseline[DIVERSITY_PAIR_KEYS + ["rollout_id", "success", "n_steps", "n_chunks"]]
                 .rename(columns={"rollout_id": "baseline_rollout_id",
                                  "success": "baseline_success",
                                  "n_steps": "baseline_n_steps",
                                  "n_chunks": "baseline_n_chunks"}),
                 on=DIVERSITY_PAIR_KEYS, validate="one_to_one"))
assert len(paired) == len(delayed), (
    f"Only {len(paired)}/{len(delayed)} delayed rows match the historical baseline")
for column in ("delayed_success", "baseline_success"):
    paired[column] = paired[column].astype(bool)
if REQUIRE_FULL_COHORT:
    assert len(paired) == EXPECTED_EPISODES, (
        f"Expected {EXPECTED_EPISODES} matched episodes, found {len(paired)}")
print({"matched_episodes": len(paired), "expected": EXPECTED_EPISODES,
       "coverage_pct": 100 * len(paired) / EXPECTED_EPISODES,
       "suites": paired.suite.nunique(),
       "policy": "chunks 0..4 unrefined; refine-last from chunk 5"})

## 3. Whole-cohort and per-suite paired results

In [ ]:
def summarize(group):
    baseline_values = group.baseline_success.to_numpy(bool)
    delayed_values = group.delayed_success.to_numpy(bool)
    lo, hi = paired_bootstrap_ci(baseline_values, delayed_values, n_boot=5000)
    f_to_s = int((~baseline_values & delayed_values).sum())
    s_to_f = int((baseline_values & ~delayed_values).sum())
    return pd.Series({
        "episodes": len(group),
        "unrefined_source_sr_pct": 100 * baseline_values.mean(),
        "delayed_refinement_sr_pct": 100 * delayed_values.mean(),
        "delayed_minus_source_pp": 100 * (delayed_values.mean() - baseline_values.mean()),
        "delta_ci_low_pp": 100 * lo,
        "delta_ci_high_pp": 100 * hi,
        "failure_to_success": f_to_s,
        "success_to_failure": s_to_f,
        "paired_p_value": discordant_test(f_to_s, s_to_f),
    })

overall = summarize(paired).to_frame().T
by_suite = pd.DataFrame([
    {"suite": suite, **summarize(group).to_dict()}
    for suite, group in paired.groupby("suite", sort=True)])
print("Primary result: every matched episode is in the SR denominator")
display(overall)
display(by_suite[["suite", "episodes", "unrefined_source_sr_pct",
                  "delayed_refinement_sr_pct", "delayed_minus_source_pp",
                  "failure_to_success", "success_to_failure"]])
paired.to_csv(OUTPUT / "matched_episodes.csv", index=False)
overall.to_csv(OUTPUT / "overall.csv", index=False)
by_suite.to_csv(OUTPUT / "by_suite.csv", index=False)

## 4. Matched success rates and per-suite change

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(by_suite)); width = .38
labels = by_suite.suite.str.removeprefix("libero_")
axes[0].bar(x - width/2, by_suite.unrefined_source_sr_pct, width,
            label="unrefined source", color="#4C78A8")
axes[0].bar(x + width/2, by_suite.delayed_refinement_sr_pct, width,
            label="refine from chunk 6", color="#F58518")
axes[0].set_xticks(x, labels, rotation=40, ha="right")
axes[0].set(ylabel="Success rate (%)", ylim=(0, 105),
            title="Delayed refinement vs matched source baseline")
axes[0].legend(); axes[0].grid(axis="y", alpha=.2)

colors = np.where(by_suite.delayed_minus_source_pp >= 0, "#54A24B", "#E45756")
axes[1].bar(x, by_suite.delayed_minus_source_pp, color=colors)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].axhline(overall.delayed_minus_source_pp.iloc[0], color="#9467BD",
                linestyle="--",
                label=f"overall: {overall.delayed_minus_source_pp.iloc[0]:+.2f} pp")
axes[1].set_xticks(x, labels, rotation=40, ha="right")
axes[1].set(ylabel="Delayed refinement minus source SR (percentage points)",
            title="Whole-matched-cohort SR change by suite")
axes[1].legend(); axes[1].grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(OUTPUT / "source_delayed_refinement.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Concise conclusion

In [ ]:
result = overall.iloc[0]
print(f"Matched episodes: {int(result.episodes)}/{EXPECTED_EPISODES}")
print(f"Unrefined source:  {result.unrefined_source_sr_pct:.2f}%")
print(f"Delayed refine:    {result.delayed_refinement_sr_pct:.2f}%")
print(f"Paired change:     {result.delayed_minus_source_pp:+.2f} pp "
      f"(95% CI {result.delta_ci_low_pp:+.2f} to {result.delta_ci_high_pp:+.2f})")
print(f"Transitions:       {int(result.failure_to_success)} F->S, "
      f"{int(result.success_to_failure)} S->F")